In [ ]:
#pip install transformers[torch] accelerate -U

In [ ]:
#pip install transformers datasets accelerate sentence-transformers torch --upgrade

In [ ]:
#pip install datasets


In [ ]:
#pip show datasets

In [ ]:
#pip install torch

In [ ]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset
import torch

# Check the availability of cuda
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# Load pretrained model
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)

# Load the tokenizer for the pretrained model
tokenizer = AutoTokenizer.from_pretrained('gpt2', use_fast=True)

# Load the "train.txt" dataset
train_dataset = load_dataset('text', data_files={'train': 'train.txt'}, split='train')

# Tokenize the dataset and create labels
def tokenize_function(examples):
    tokenized_output = tokenizer(examples['text'], truncation=True, max_length=128)
    # Filter out sequences that are empty after tokenization
    non_empty = [len(ids) > 0 for ids in tokenized_output['input_ids']]
    tokenized_output = {k: [v[i] for i in range(len(v)) if non_empty[i]] for k, v in tokenized_output.items()}
    # Labels are identical to input_ids for causal LM
    tokenized_output['labels'] = tokenized_output['input_ids'].copy()
    return tokenized_output

tokenized_datasets = train_dataset.map(tokenize_function, batched=True, remove_columns=['text'])

# Define training arguments with `remove_unused_columns=False`
training_args = TrainingArguments(
    output_dir='./fine_tuned_gpt2',
    overwrite_output_dir=True,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    fp16=torch.cuda.is_available(),
    logging_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    remove_unused_columns=False  # Prevents removing 'labels' or other columns
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets,
)

# Start training
trainer.train()

# Save the fine-tuned model and tokenizer
trainer.save_model('./fine_tuned_gpt2')
tokenizer.save_pretrained('./fine_tuned_gpt2')
